In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/student-performance-factors/StudentPerformanceFactors.csv


# Imports + Initial Impressions

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/kaggle/input/student-performance-factors/StudentPerformanceFactors.csv")

df.head()
df.shape
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


From an initial glance, we can see that the average exam score from this dataset is 67, and σ is approximately 3.89. So without any outlier mitigation, 99.7% of the exam scores fall within the approximate range of [55.56, 78.9]

# Preprocessing

First, we'll check for any missing or improperly formatted values in our dataset.

In [3]:
df.isna().sum().sort_values(ascending=False)

Parental_Education_Level      90
Teacher_Quality               78
Distance_from_Home            67
Hours_Studied                  0
Attendance                     0
Gender                         0
Learning_Disabilities          0
Physical_Activity              0
Peer_Influence                 0
School_Type                    0
Family_Income                  0
Tutoring_Sessions              0
Internet_Access                0
Motivation_Level               0
Previous_Scores                0
Sleep_Hours                    0
Extracurricular_Activities     0
Access_to_Resources            0
Parental_Involvement           0
Exam_Score                     0
dtype: int64

Parental_Educational_Level and Teacher_Quality are both important features, so we'll simply drop records that are missing those values. I didn't plan on using Distance_from_Home in the analysis, so we'll allow records with that attribute as blank to stay. Overall, it's a nearly complete dataset.

In [4]:
df = df.dropna(subset=["Parental_Education_Level"])
df = df.dropna(subset=["Teacher_Quality"])

Next, we'll want to check for each unique value in "Gender" to prepare the column for normalization.

In [5]:
df["Gender"].unique()

array(['Male', 'Female'], dtype=object)

Let's check for any potential class imbalance in the 'Gender' column

In [6]:
print("Number of male respondents: " + str((df["Gender"] == "Male").sum()))
print("Number of female respondents: " + str((df["Gender"] == "Female").sum()))

Number of male respondents: 3723
Number of female respondents: 2720


We'll probably need to over or undersample after creating our test and train sets.

Now, we'll perform a simple statistical analysis of the 'Sleep_Hours' category, because it's our target variable.

In [11]:
df['Sleep_Hours'].describe()

count    6443.000000
mean        7.032594
std         1.467814
min         4.000000
25%         6.000000
50%         7.000000
75%         8.000000
max        10.000000
Name: Sleep_Hours, dtype: float64

# Inquiry: What is the relationship between sleep and exam scores?

In [10]:
df_sleep = df['Sleep_Hours']
df_exam_score = df['Exam_Score']


